# EMA 6938 - Data Science for Materials
## Week 11 Notebook: ML Case Studies Across Material Classes

**Name:** *(your name here)*  
**Date:** *(date)*  
**Kernel:** Python (matds)

---

**Chapters:** Sandfeld Ch. 17 + Instructor Case Studies  
**Format:** Due **Sunday 11:59 PM**  
**Dataset:** Choose one: `week11_steel_yield.csv` / `week11_ceramic_modulus.csv` / `week11_polymer_tg.csv`

---

### How to use this notebook
- **Demo cells** (`# LECTURE DEMO`) reproduce examples from the lecture. Run them, understand them.
- **Task cells** (`# YOUR CODE HERE`) require you to write code.
- **Reflection cells** require written markdown answers. Replace the italic placeholder text.
- 
This notebook has 6 parts:

| Part | Title | Connects to |
|------|-------|-------------|
| A | Load & Choose Your Dataset | Lecture 1, Segment 1 |
| B | Feature Engineering (class-specific) | Lectures 1–2 |
| C | Leakage-Aware Split & Baseline | Lecture 1, Segment 2 |
| D | Tune & Evaluate | Lecture 1, Segment 3 |
| E | Feature Importance & Physical Interpretation | All lectures |
| F | Reflection & Final Project Connection | All lectures |

Choose the track closest to your own research area/interests/occupation - this directly prepares your final project.

---

**Submission:** Upload this `.ipynb` file to Canvas. Run `Kernel → Restart & Run All` before submitting to confirm all cells execute cleanly.

> **AI tool disclosure:** If you used any AI coding assistant (GitHub Copilot, ChatGPT, etc.) while completing this notebook, describe briefly which tool, for what purpose, and what you verified yourself. Delete this line if no AI tools were used.

In [ ]:
# Cell 0 — Environment check
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (GroupKFold, LeaveOneOut,
    RandomizedSearchCV, cross_val_score, learning_curve)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

# XGBoost and SHAP (Week 10 — Part D3 and E1b)
try:
    import xgboost as xgb
    print(f'XGBoost {xgb.__version__} OK')
except ImportError:
    print('XGBoost not found — install: pip install xgboost')
try:
    import shap
    print(f'SHAP {shap.__version__} OK')
except ImportError:
    print('SHAP not found — install: pip install shap')

# Polymers track only - will be skipped if rdkit not installed
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Draw
    from rdkit import DataStructs
    print("RDKit OK")
except ImportError:
    print("RDKit not found - only needed for polymers track. Install: conda install -c conda-forge rdkit")

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
print("Core imports OK")

---
## Part A - Load & Choose Your Dataset

### A1: Load your chosen dataset

> **Edit the line below to load your chosen track (metals / ceramics / polymers).**

In [ ]:
# Cell A1 -
# LECTURE DEMO
# ── EDIT THIS LINE ──────────────────────────────────────────────────────────
TRACK = 'metals'   # Options: 'metals', 'ceramics', 'polymers'
# ────────────────────────────────────────────────────────────────────────────

file_map = {
    'metals':   'data/week11_steel_yield.csv',
    'ceramics': 'data/week11_ceramic_modulus.csv',
    'polymers': 'data/week11_polymer_Tg.csv',
}
target_map = {
    'metals':   'yield strength',
    'ceramics': 'shear_modulus',
    'polymers': 'Tg',
}
group_map = {
    'metals':   'alloy_family',
    'ceramics': 'ceramic_type',
    'polymers': 'monomer_family',
}

df = pd.read_csv(file_map[TRACK])

# For metals csv
if TRACK == 'metals':
    # Group by alloy complexity — count elements with wt% > 0.01
    # This produces 4 balanced groups across all 312 steels regardless of Cr distribution
    alloying_cols = ['c','mn','si','cr','ni','mo','v','n','nb','co','w','al','ti']
    df['n_alloying'] = (df[alloying_cols] > 0.01).sum(axis=1)
    df['alloy_family'] = pd.cut(df['n_alloying'],
    bins=[0, 2, 4, 6, 20],
    labels=['simple','low_alloy','medium_alloy','complex'],
    include_lowest=True)
    print(f"Alloy family distribution:")
    print(df['alloy_family'].value_counts().sort_index())

# For polymer csv
if TRACK == 'polymers':
    def smiles_to_family(s):
        if 'c1ccccc1' in s: return 'aromatic'
        if 'C(=O)O' in s:   return 'ester'
        if 'C#N' in s:       return 'nitrile'
        if 'C(=O)N' in s:   return 'amide'
        if 'C=C' in s:        return 'vinyl'
        if 'Si' in s:         return 'silicone'
        if 'C1CCCCC1' in s:  return 'aliphatic_ring'
        return 'other'
    df['monomer_family'] = df['SMILES'].apply(smiles_to_family)  # SMILES column — uppercase


# For ceramics csv — derive ceramic_type grouping column from formula
if TRACK == 'ceramics':
    try:
        from pymatgen.core import Composition as _Comp
        def ceramic_type(formula):
            try:
                elems = [str(e) for e in _Comp(formula).elements]
                if 'O' in elems: return 'oxide'
                if 'N' in elems: return 'nitride'
                if 'C' in elems: return 'carbide'
                if 'S' in elems: return 'sulfide'
                return 'other'
            except: return 'other'
        df['ceramic_type'] = df['formula'].apply(ceramic_type)
    except ImportError:
        # Fallback: simple string match on formula
        def ceramic_type_str(formula):
            f = str(formula)
            if 'O' in f: return 'oxide'
            if 'N' in f: return 'nitride'
            if 'C' in f: return 'carbide'
            if 'S' in f: return 'sulfide'
            return 'other'
        df['ceramic_type'] = df['formula'].apply(ceramic_type_str)

# Establishing targets and groups
TARGET = target_map[TRACK]
GROUP_COL = group_map[TRACK]

print(f"Track: {TRACK.upper()}")
print(f"Dataset shape: {df.shape}")
print(f"Target: {TARGET}")
print(f"\nTarget statistics:")
print(df[TARGET].describe().round(3))
print(f"\nGroup key ('{GROUP_COL}') - unique values: {df[GROUP_COL].nunique()}")
print(df[GROUP_COL].value_counts().head(8))

### A2: Target property distribution

In [ ]:
# Cell A2
# LECTURE DEMO
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(df[TARGET], bins=40, color='#1C2B4A', alpha=0.85, edgecolor='white', density=True)
axes[0].set_xlabel(TARGET); axes[0].set_title('Raw distribution')

axes[1].hist(np.log1p(df[TARGET].clip(lower=0.001)), bins=40, color='#0D9488', alpha=0.85, edgecolor='white', density=True)
axes[1].set_xlabel(f'log({TARGET}+1)'); axes[1].set_title('Log-transformed')

stats.probplot(df[TARGET], dist='norm', plot=axes[2])
axes[2].set_title('Q-Q plot (normal)')

plt.suptitle(f'{TRACK.capitalize()} - {TARGET} distribution', fontsize=11)
plt.tight_layout()
plt.savefig('A2_distribution.png', dpi=150)
plt.show()

# Normality test
stat, p = stats.shapiro(df[TARGET].sample(min(50, len(df)), random_state=SEED))
print(f"Shapiro-Wilk p-value: {p:.4f}  ({'normal' if p>0.05 else 'non-normal'} distribution)")

### A3: Task - CV group key justification

> Identify the group key for GroupKFold and justify it physically.

In [ ]:
# Cell A3
# LECTURE DEMO
print(f"CV Group key: '{GROUP_COL}'")
print(f"Number of unique groups: {df[GROUP_COL].nunique()}")
print(f"Group sizes:")
print(df.groupby(GROUP_COL).size().sort_values(ascending=False).head(10))

**Reflection A3 - fill in this cell:**

Why is `GROUP_COL` the appropriate key for GroupKFold in your chosen material class?
What type of composition or structural leakage does this grouping prevent?
What would happen to the reported R2 if you used a random split instead?

*Your answer here*

---
## Part B - Feature Engineering (class-specific)

### B1: Set up the feature matrix X

In [ ]:
# Cell B1
# LECTURE DEMO
if TRACK == 'metals':
    # Elemental composition (wt%) + processing parameters
    exclude = [TARGET, GROUP_COL, 'alloy_id', 'alloy_name']
    feature_cols = [c for c in df.columns if c not in exclude and df[c].dtype in ['float64','float32','int64']]
    X_raw = df[feature_cols].fillna(df[feature_cols].median()).values

elif TRACK == 'ceramics':
    # Oxide mole fractions + sintering parameters
    exclude = [TARGET, GROUP_COL, 'formula', 'bulk_modulus']
    feature_cols = [c for c in df.columns if c not in exclude and df[c].dtype in ['float64','float32','int64']]
    X_raw = df[feature_cols].fillna(df[feature_cols].median()).values

elif TRACK == 'polymers':
    # Morgan fingerprints from SMILES
    _fp_generator = AllChem.GetMorganGenerator(radius=2, fpSize=1024)

    def smiles_to_fp(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None: return None
            return np.array(_fp_generator.GetFingerprintAsNumPy(mol))
        except: return None

    n_total = len(df)   # store BEFORE the valid filter
    fps = df['SMILES'].apply(smiles_to_fp)
    valid = fps.notna()
    df = df[valid].reset_index(drop=True)
    fps = fps[valid]
    X_raw = np.vstack(fps.values)
    feature_cols = [f'bit_{i}' for i in range(X_raw.shape[1])]
    print(f"Successfully featurised: {len(df):,} / {n_total:,} polymers")

y = df[TARGET].values
groups = df[GROUP_COL].values

print(f"X shape: {X_raw.shape}")
print(f"y shape: {y.shape}")
print(f"Feature count: {len(feature_cols)}")

### B2: Standardise features (pipeline will handle this per-fold, but check scaling here)

In [ ]:
# Cell B2 - Quick check on one fold
# LECTURE DEMO
n_groups = len(np.unique(groups))
n_splits = min(5, n_groups)
gkf = GroupKFold(n_splits=n_splits)
train_idx, test_idx = list(gkf.split(X_raw, y, groups))[0]
print(f"Unique groups: {n_groups} - using GroupKFold(n_splits={n_splits})")

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_raw[train_idx])
print(f"Training fold scaled - mean: {X_tr_sc.mean():.4f}  std: {X_tr_sc.std():.4f}")
print(f"Training fold size: {len(train_idx)}  Test fold size: {len(test_idx)}")

### B3: Task - correlation filter (metals and ceramics tracks)

In [ ]:
# Cell B3
# LECTURE DEMO
if TRACK in ['metals', 'ceramics']:
    X_df = pd.DataFrame(X_raw[train_idx], columns=feature_cols)
    corr_upper = X_df.corr().abs().where(np.triu(np.ones(X_df.shape[1], dtype=bool), k=1))
    cols_to_drop = [c for c in corr_upper.columns if any(corr_upper[c] > 0.95)]
    feature_cols_filtered = [c for c in feature_cols if c not in cols_to_drop]
    print(f"Removed by correlation filter: {len(cols_to_drop)} features")
    print(f"Remaining: {len(feature_cols_filtered)} features")
    X_raw_filt = X_raw[:, [feature_cols.index(c) for c in feature_cols_filtered]]
else:
    # Polymers: fingerprint bits are already binary — correlation filter not applicable
    X_raw_filt = X_raw
    feature_cols_filtered = feature_cols
    print(f"Polymers track: skipping correlation filter (binary fingerprint bits)")
    print(f"Feature count: {len(feature_cols_filtered)}")

---
## Part C - Leakage-Aware Split & Baseline

### C1: GroupKFold split (or LOO for small ceramic datasets)

In [ ]:
# Cell C1
# LECTURE DEMO
if TRACK == 'ceramics' and len(df) < 400:  # threshold raised to 400 for 300-entry ceramic dataset
    print(f"Small ceramics dataset (n={len(df)}) - using LeaveOneOut CV")
    cv_strategy = LeaveOneOut()
    CV_NAME = 'LOO'
else:
    n_groups = len(np.unique(groups))
    n_splits = min(5, n_groups)
    cv_strategy = GroupKFold(n_splits=n_splits)
    CV_NAME = f'GroupKFold({n_splits})'
    print(f"Using {CV_NAME} with group key '{GROUP_COL}'")

# Extract fold 0 for Parts D-E
if hasattr(cv_strategy, 'split') and TRACK != 'ceramics' or len(df) >= 200:
    fold0 = list(GroupKFold(n_splits).split(X_raw_filt, y, groups))[0]
else:
    fold0 = list(LeaveOneOut().split(X_raw_filt, y))[0]

tr_idx, te_idx = fold0
X_train, X_test = X_raw_filt[tr_idx], X_raw_filt[te_idx]
y_train, y_test = y[tr_idx], y[te_idx]
groups_train = groups[tr_idx]
print(f"Fold 0: train {len(X_train)}, test {len(X_test)}")
print(np.unique(groups_train))

### C2: Compare random split vs. group CV - leakage gap

In [ ]:
# Cell C2
# LECTURE DEMO
pipe = Pipeline([("sc", StandardScaler()),
                 ("rf", RandomForestRegressor(100, n_jobs=-1, random_state=SEED))])

r2_random = cross_val_score(pipe, X_raw_filt, y, cv=n_splits, scoring='r2')
mae_random = cross_val_score(pipe, X_raw_filt, y, cv=n_splits, scoring='neg_mean_absolute_error')

if CV_NAME.startswith('Group'):
    r2_group = cross_val_score(pipe, X_raw_filt, y,
                            cv=GroupKFold(n_splits), groups=groups, scoring='r2')
    print(f"Random 5-fold R2:  {r2_random.mean():.3f} ± {r2_random.std():.3f}")
    print(f"GroupKFold R2:     {r2_group.mean():.3f}  ± {r2_group.std():.3f}")
    print(f"Leakage gap:       {r2_random.mean()-r2_group.mean():.3f}")
else:
    loo_scores = cross_val_score(pipe, X_raw_filt, y, cv=LeaveOneOut(), scoring='neg_mean_absolute_error')
    print(f"Random 5-fold MAE:  {-mae_random.mean():.3f} ± {mae_random.std():.3f}")
    print(f"LOO MAE: {-loo_scores.mean():.3f}  (std not comparable to k-fold — 1 sample per fold)")
    print(f"Leakage gap (MAE):  {-loo_scores.mean() - (-mae_random.mean()):.3f}  "
      f"(positive = LOO harder than random split)")

### C3: Default RF baseline on fold 0

In [ ]:
# Cell C3
# LECTURE DEMO
sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_train)
X_te_sc = sc.transform(X_test)

rf_default = RandomForestRegressor(100, n_jobs=-1, random_state=SEED)
rf_default.fit(X_tr_sc, y_train)
y_pred_def = rf_default.predict(X_te_sc)

r2_def  = r2_score(y_test, y_pred_def)
mae_def = mean_absolute_error(y_test, y_pred_def)
print(f"Default RF baseline:  R2={r2_def:.3f}  MAE={mae_def:.3f}")

---
## Part D - Tune & Evaluate

### D1: RandomizedSearchCV with class-appropriate inner CV

In [ ]:
# Cell D1
# LECTURE DEMO
param_dist = {
    "rf__n_estimators":     [100, 200, 300],
    "rf__max_depth":        [None, 5, 10, 15],
    "rf__min_samples_leaf": [1, 2, 5],
    "rf__max_features":     ["sqrt", "log2", 0.3],
}

inner_cv = GroupKFold(len(np.unique(groups_train))) if CV_NAME.startswith('Group') else LeaveOneOut()

rscv = RandomizedSearchCV(
    pipe, param_dist, n_iter=20,
    cv=inner_cv,
    scoring='r2' if CV_NAME.startswith('Group') else 'neg_mean_absolute_error', n_jobs=-1, random_state=SEED, verbose=0
)
rscv.fit(X_train, y_train, groups=groups_train if CV_NAME.startswith('Group') else None)

print(f"Best params: {rscv.best_params_}")
metric_label = 'R²' if CV_NAME.startswith('Group') else 'neg-MAE (= −MAE in GPa)'
print(f"Best CV score ({metric_label}): {rscv.best_score_:.3f}")

### D2: Tuned RF - test fold evaluation

In [ ]:
# Cell D2
# LECTURE DEMO
y_pred_tuned = rscv.predict(X_test)
r2_tuned   = r2_score(y_test, y_pred_tuned)
mae_tuned  = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

comparison = pd.DataFrame({
    'Model': ['Default RF', 'Tuned RF'],
    'R2':    [round(r2_def,3), round(r2_tuned,3)],
    'MAE':   [round(mae_def,3), round(mae_tuned,3)],
})
print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred_tuned, s=20, alpha=0.7, color='#0D9488')
lim = [min(y_test.min(), y_pred_tuned.min())-5,
       max(y_test.max(), y_pred_tuned.max())+5]
ax.plot(lim, lim, 'k--', lw=0.8)
ax.set_xlabel(f'Actual {TARGET}'); ax.set_ylabel('Predicted')
ax.set_title(f'Tuned RF - {TRACK.capitalize()}  $R^2$={r2_tuned:.3f}')
plt.tight_layout(); plt.savefig('D2_prediction_scatter.png', dpi=150); plt.show()

### D3: XGBoost - compare against tuned RF

XGBoost (eXtreme Gradient Boosting) is a sequentially-built ensemble:
each tree corrects the residuals of the previous one, rather than averaging
independent trees (as in Random Forest). For small-n datasets (ceramics)
it often outperforms RF because it focuses learning power on the hardest
examples. For large datasets it is also faster than RF per tree.

> **Why compare here?** The midterm Extension 1 asked you to replace RF with
> XGBoost - this cell shows the same comparison in the context of the three
> material classes, so you can see whether the improvement is consistent
> across metals, ceramics, and polymers.

In [ ]:
# Cell D3 - XGBoost vs. Tuned RF
# LECTURE DEMO
try:
    import xgboost as xgb
except ImportError:
    raise ImportError("XGBoost not installed. Run: pip install xgboost")

xgb_pipe = Pipeline([
    ("sc", StandardScaler()),
    ("xgb", xgb.XGBRegressor(
        n_estimators    = 300,
        max_depth       = 6,
        learning_rate   = 0.05,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        random_state    = SEED,
        verbosity       = 0,
    ))
])
xgb_pipe.fit(X_train, y_train)
y_pred_xgb = xgb_pipe.predict(X_test)
r2_xgb  = r2_score(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

# Full comparison table
comparison_full = pd.DataFrame({
    'Model': ['Default RF', 'Tuned RF (RandomizedSearch)', 'XGBoost (default)'],
    'R2':    [round(r2_def,3),   round(r2_tuned,3), round(r2_xgb,3)],
    'MAE':   [round(mae_def,3),  round(mae_tuned,3), round(mae_xgb,3)],
})
print(comparison_full.to_string(index=False))
print(f"\nXGBoost vs Tuned RF: ΔR² = {r2_xgb - r2_tuned:+.3f}")

# Parity plot side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, preds, label, col in zip(
    axes,
    [y_pred_tuned, y_pred_xgb],
    ['Tuned RF', 'XGBoost'],
    ['#0D9488', '#7C3AED']
):
    ax.scatter(y_test, preds, s=20, alpha=0.7, color=col)
    lim = [min(y_test.min(), preds.min())-5,
           max(y_test.max(), preds.max())+5]
    ax.plot(lim, lim, 'k--', lw=0.8)
    ax.set_xlabel(f'Actual {TARGET}')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{label}  $R^2$={r2_score(y_test, preds):.3f}')
plt.suptitle(f'{TRACK.capitalize()} - RF vs XGBoost', fontsize=11)
plt.tight_layout()
plt.savefig('D3_xgb_comparison.png', dpi=150)
plt.show()

### D4: Learning curve

In [ ]:
# Cell D4
# LECTURE DEMO
lc_scoring = 'r2' if CV_NAME.startswith('Group') else 'neg_mean_absolute_error'

train_sizes, tr_scores, cv_scores = learning_curve(
    rscv.best_estimator_, X_train, y_train,
    train_sizes=np.linspace(0.15, 1.0, 7),
    cv=GroupKFold(len(np.unique(groups_train))) if CV_NAME.startswith('Group') else LeaveOneOut(),
    groups=groups_train if CV_NAME.startswith('Group') else None,
    scoring=lc_scoring, n_jobs=-1
)

fig, ax = plt.subplots(figsize=(5, 5))
metric = 'R²' if lc_scoring == 'r2' else 'neg-MAE'
ax.set_ylabel(metric)
ax.plot(train_sizes, tr_scores.mean(1), 'o-', color='#1C2B4A', lw=2, label=f'Train {metric}')
ax.plot(train_sizes, cv_scores.mean(1), 'o-', color='#0D9488', lw=2, label=f'CV {metric} ({CV_NAME})')

gap = tr_scores.mean(1)[-1] - cv_scores.mean(1)[-1]
print(f"Final train-CV gap ({metric}): {gap:.3f}")

#ax.fill_between(train_sizes, tr_scores.mean(1)-tr_scores.std(1), tr_scores.mean(1)+tr_scores.std(1), alpha=0.12, color='#1C2B4A')
#ax.fill_between(train_sizes, cv_scores.mean(1)-cv_scores.std(1), cv_scores.mean(1)+cv_scores.std(1), alpha=0.12, color='#0D9488')
ax.set_xlabel('Training set size'); ax.set_ylabel('R2')
ax.set_title(f'Learning Curve - {TRACK.capitalize()}')
ax.legend(); plt.tight_layout()
plt.savefig('D4_learning_curve.png', dpi=150); plt.show()

---
## Part E - Feature Importance & Physical Interpretation

### E1: Top 15 feature importances

In [ ]:
# Cell E1
# LECTURE DEMO
best_rf = rscv.best_estimator_.named_steps['rf']
importances = pd.Series(best_rf.feature_importances_, index=feature_cols_filtered)
top15 = importances.nlargest(15)

fig, ax = plt.subplots(figsize=(8, 6))
top15.sort_values().plot(kind='barh', color='#0D9488', ax=ax)
ax.set_title(f'Top 15 Features - {TRACK.capitalize()} RF')
ax.set_xlabel('Mean impurity decrease')
plt.tight_layout(); plt.savefig('E1_importance.png', dpi=150); plt.show()
print(top15.round(4))

### E1b: SHAP values - model-agnostic feature importance

RF's built-in `.feature_importances_` (mean impurity decrease) has a known
bias: it overweights high-cardinality features and doesn't show *direction*
(whether a high feature value increases or decreases the prediction). SHAP
(SHapley Additive exPlanations) fixes both problems:

- **Direction**: positive SHAP = feature value pushed prediction up; negative = pushed down
- **Magnitude**: same units as the target (MPa, GPa, °C)
- **Per-sample**: you can see why the model made a specific prediction, not just
  which features matter on average

> **Note:** SHAP `TreeExplainer` works directly on tree-based models (RF, XGBoost)
> without any approximation - it's exact and fast for these model classes.

In [ ]:
# Cell E1b — SHAP values vs RF importances
# LECTURE DEMO
try:
    import shap
except ImportError:
    raise ImportError("SHAP not installed. Run: pip install shap")

# Use the tuned RF from D1 (best_rf = rscv.best_estimator_.named_steps['rf'])
# We explain predictions on X_test (already scaled in pipe — use raw for SHAP)
# Scale X_test using the fitted scaler from the tuned pipeline
scaler_fitted = rscv.best_estimator_.named_steps['sc']
X_test_scaled = scaler_fitted.transform(X_test)

# Subsample X_test for SHAP — exact TreeExplainer is slow on large test sets
# 200 samples is enough to get stable mean |SHAP| rankings
N_SHAP = min(200, len(X_test))
rng = np.random.default_rng(SEED)
shap_idx = rng.choice(len(X_test), size=N_SHAP, replace=False)
X_test_shap = X_test_scaled[shap_idx]

# Actually X_test is already unscaled — just pass it directly
# Use 50 background samples for interventional SHAP
background = shap.sample(X_test_shap, 50, random_state=SEED)

explainer   = shap.TreeExplainer(
    rscv.best_estimator_.named_steps['rf'],
    data=background,
    feature_perturbation='interventional'
)
shap_values = explainer.shap_values(X_test_shap, check_additivity=False)

# ── Plot 1: SHAP summary (beeswarm) ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: RF importance (E1 reproduced for comparison)
top15_idx = importances.nlargest(15).index
top15_vals = importances[top15_idx].sort_values()
axes[0].barh(top15_vals.index, top15_vals.values, color='#0D9488')
axes[0].set_title('RF Feature Importance\n(mean impurity decrease - unsigned)')
axes[0].set_xlabel('Importance')

# Right: SHAP mean absolute values for same features
shap_df = pd.DataFrame(np.abs(shap_values),
                        columns=feature_cols_filtered[:shap_values.shape[1]])
shap_mean = shap_df.mean().nlargest(15).sort_values()
axes[1].barh(shap_mean.index, shap_mean.values, color='#7C3AED')
axes[1].set_title(f'SHAP Mean |value|\n(units: {TARGET})')
axes[1].set_xlabel(f'Mean |SHAP value| ({TARGET})')

plt.suptitle(f'{TRACK.capitalize()} - RF importance vs. SHAP', fontsize=11)
plt.tight_layout()
plt.savefig('E1b_shap_comparison.png', dpi=150)
plt.show()

# ── Text comparison ───────────────────────────────────────────────────────
print("Top-5 by RF importance:  ", importances.nlargest(5).index.tolist())
print("Top-5 by SHAP magnitude: ", shap_df.mean().nlargest(5).index.tolist())
print()
rf_top5   = set(importances.nlargest(5).index)
shap_top5 = set(shap_df.mean().nlargest(5).index)
overlap   = rf_top5 & shap_top5
print(f"Features in both top-5 lists: {overlap}")
print(f"Agreement: {len(overlap)}/5 features")

### E2: Polymers only - decode top-3 fingerprint bits

> Skip this cell if you chose metals or ceramics.

In [ ]:
# Cell E2
# LECTURE DEMO
if TRACK == 'polymers':
    # Take the actual top-3 most important bits (largest importance values)
    top3_bits = [int(f.split('_')[1]) for f in importances.nlargest(3).index]
    print("Top 3 most important fingerprint bits:", top3_bits)

    # Find a molecule that activates all three bits
    generator = AllChem.GetMorganGenerator(radius=2, fpSize=1024)
    sample_smiles = None

    for smiles in df['SMILES']:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
        fp = np.array(generator.GetFingerprintAsNumPy(mol))
        if all(fp[b] == 1 for b in top3_bits):
            sample_smiles = smiles
            break

    if sample_smiles is None:
        # No single molecule activates all three — find one that activates the top bit
        for smiles in df['SMILES']:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue
            fp = np.array(generator.GetFingerprintAsNumPy(mol))
            if fp[top3_bits[0]] == 1:
                sample_smiles = smiles
                break

    if sample_smiles is None:
        print("Could not find a representative molecule — try increasing the search range")
    else:
        print(f"Using molecule: {sample_smiles}")
        mol = Chem.MolFromSmiles(sample_smiles)

        # Decode bit environments using new RDKit API (replaces deprecated bitInfo)
        ao = AllChem.AdditionalOutput()
        ao.CollectBitInfoMap()
        generator.GetFingerprint(mol, additionalOutput=ao)
        bit_info = ao.GetBitInfoMap()

        for bit in top3_bits:
            if bit in bit_info:
                print(f"\nBit {bit}: appears in {len(bit_info[bit])} atom environments")
                print(f"  Atom indices and radii: {list(bit_info[bit])[:3]}")
            else:
                print(f"\nBit {bit}: not active in this molecule")
else:
    print(f"Skipping fingerprint decoding — {TRACK} track does not use Morgan fingerprints")

### E3: Task - physical interpretation of top-3 features

In [ ]:
# Cell E3
# TASK CELL
print("Top 3 most important features:")
for i, (feat, imp) in enumerate(top15.head(3).items(), 1):
    print(f"  {i}. {feat}  (importance = {imp:.4f})")

# YOUR TASK: Fill in the reflection cell below
# For each feature: (a) name it, (b) physical property it encodes, (c) mechanism linking it to target

**Reflection E3 - fill in this cell:**

For each of your top 3 features:

**Feature 1 - [name]:**
- Physical property encoded:
- Physical mechanism linking to target (yield strength / K1c / Tg):

**Feature 2 - [name]:**
- Physical property encoded:
- Physical mechanism:

**Feature 3 - [name]:**
- Physical property encoded:
- Physical mechanism:

*Your answer here*

---
## Part F - Reflection & Final Project Connection

### F1: What is the biggest ML challenge for your material class?

In 3–4 sentences: What is the single biggest challenge of applying ML to your chosen material class compared to the MP-oxide datasets from Weeks 5–10?

Is it:
- **n** - too few samples for reliable CV?
- **Feature representation** - MAGPIE doesn't apply, need fingerprints or processing features?
- **CV strategy** - random splits give very different results from group CV?
- **Target distribution** - non-Gaussian (Weibull for ceramics, multimodal for alloys)?

Give a specific example from your results above.

*Your answer here*

### F2: Final project connection

In 3–4 sentences: How does your chosen material class in this notebook connect to your final project?

Answer specifically:
1. What material class does your final project use - metals, ceramics, polymers, or something else?
2. Which feature type will you use: MAGPIE composition, processing parameters, Morgan fingerprints, or another representation?
3. What CV group key will you use to prevent leakage in your final project dataset?

*Your answer here*

---
## Day 2 Session

### Demo 1 - Active learning acquisition functions compared

In [ ]:
# DEMO 1 — Compare acquisition functions: uncertainty vs. UCB
# LECTURE DEMO

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# Simulate: train on current dataset, predict mean and uncertainty for candidates
rf_demo = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
rf_demo.fit(X_train, y_train)

# RF uncertainty = std of individual tree predictions
tree_preds = np.stack([tree.predict(X_test) for tree in rf_demo.estimators_], axis=1)
mean_pred  = tree_preds.mean(axis=1)
std_pred   = tree_preds.std(axis=1)

# Acquisition functions
ucb_beta    = 2.0
acq_uncertainty = std_pred                    # pure exploration
acq_ucb         = mean_pred + ucb_beta * std_pred  # exploitation + exploration

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, acq, label, color in zip(
        axes,
        [acq_uncertainty, acq_ucb],
        ['Uncertainty sampling (pure exploration)',
         f'UCB (β={ucb_beta}) - balance explore/exploit'],
        ['#0D9488', '#7C3AED']):
    top_idx = np.argsort(acq)[::-1][:20]
    ax.scatter(y_test, mean_pred, s=8, alpha=0.3, color='#CBD5E1')
    ax.scatter(y_test[top_idx], mean_pred[top_idx],
               s=60, color=color, alpha=0.9, zorder=5,
               label=f'Top 20 candidates')
    ax.set_xlabel(TARGET); ax.set_ylabel('Predicted')
    ax.set_title(label, fontsize=10); ax.legend(fontsize=8)
plt.suptitle('Active learning: which candidates to measure next?', fontsize=11)
plt.tight_layout()
plt.savefig('Day2_acquisition_functions.png', dpi=150, bbox_inches='tight')
plt.show()

### Demo 2 - Uncertainty calibration check

In [ ]:
# DEMO 2 - Is the RF uncertainty estimate well-calibrated?
# LECTURE DEMO

import numpy as np
import matplotlib.pyplot as plt

actual_errors = np.abs(y_pred_tuned - y_test)  # from Part D

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: predicted std vs. actual absolute error
axes[0].scatter(std_pred, actual_errors, s=8, alpha=0.35, color='#1C2B4A')
lim = max(std_pred.max(), actual_errors.max())
axes[0].plot([0, lim], [0, lim], 'r--', lw=1.2, label='Perfect calibration')
axes[0].set_xlabel('Predicted uncertainty (RF std)  eV')
axes[0].set_ylabel('Actual |error|  eV')
axes[0].set_title('Uncertainty calibration')
axes[0].legend()

# Binned: mean actual error per uncertainty decile
bins    = np.percentile(std_pred, np.linspace(0, 100, 11))
bin_idx = np.digitize(std_pred, bins) - 1
bin_idx = np.clip(bin_idx, 0, 9)
bin_means_unc = [std_pred[bin_idx==b].mean() for b in range(10)]
bin_means_err = [actual_errors[bin_idx==b].mean() for b in range(10)]

axes[1].plot(bin_means_unc, bin_means_err, 'o-', color='#0D9488', lw=2)
axes[1].plot([0, max(bin_means_unc)], [0, max(bin_means_unc)], 'r--', lw=1.2)
axes[1].set_xlabel('Mean predicted uncertainty per bin  eV')
axes[1].set_ylabel('Mean actual error per bin  eV')
axes[1].set_title('Binned calibration (10 deciles)')

plt.suptitle('RF uncertainty calibration check', fontsize=11)
plt.tight_layout()
plt.savefig('Day2_uncertainty_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

r = np.corrcoef(std_pred, actual_errors)[0,1]
print(f"Pearson r (predicted uncertainty vs. actual error): {r:.3f}")
print("Well-calibrated: r close to 1.0  |  Overconfident: r < 0.5")

### Demo 3 - Final project pipeline map
**Discussion - no live code**

This session maps the full Weeks 1–11 pipeline to your final project.
Use this checklist to confirm your project has all required components:

| Week | What you learned | What your project needs |
|------|-----------------|------------------------|
| 1–2  | Data acquisition | Your dataset loaded from a real source (not course CSV) |
| 3    | Statistical characterisation | Distribution plot of your target property |
| 4    | EDA | Correlation heatmap, at least one EDA finding |
| 5–6  | Baseline model | RF + linear model, both results reported |
| 7    | Classification (if applicable) | If target is binary, use a classifier |
| 8    | UMAP | Composition space visualisation with cluster annotation |
| 9    | Leakage-free CV | GroupKFold with a physically justified group key |
| 10   | Class-specific features | Justified feature choice for your material class |

**The minimum viable project:** Weeks 4–10 applied to your own dataset.
The UMAP from Week 9 must show your composition space.
The GroupKFold from Week 10 must be your primary CV strategy.
The baseline RF from Week 5 must be present and compared to at least one other model.

**Day 2 Discussion questions:**

1. From Demo 1: for your final project, would you use uncertainty sampling (pure exploration) or UCB (exploit + explore) as your acquisition function? What does your answer depend on? The cost of synthesis, the range of your target property, or something else?

2. From Demo 2: is the RF uncertainty well-calibrated for your material class? If the predicted uncertainty consistently underestimates actual errors, what does that tell you about whether your model is extrapolating?

3. Pipeline check: map your final project to the table above. Which week's component is missing or weakest? That is the part to strengthen before the draft is due.

---
## Submission Checklist

Before submitting, confirm all cells have been executed:

- [ ] A1: Correct track selected and dataset loaded
- [ ] A2: Distribution plot saved (`A2_distribution.png`)
- [ ] A3: Group key reflection filled in
- [ ] B1: Feature matrix set up (class-appropriate method)
- [ ] B3: Correlation filter applied and reported
- [ ] C2: Random vs. group CV comparison printed (leakage gap quantified)
- [ ] C3: Default RF baseline reported
- [ ] D1: RandomizedSearchCV best params reported
- [ ] D2: Prediction scatter saved (`D2_prediction_scatter.png`)
- [ ] D3: Learning curve saved (`D3_learning_curve.png`)
- [ ] E1: Feature importance plot saved (`E1_importance.png`)
- [ ] E2 (polymers only): Top-3 fingerprint bits decoded
- [ ] E3: Physical interpretation of top-3 features filled in
- [ ] F1–F2: Both reflection cells answered
- [ ] All reflection cells answered (no placeholder text)
- [ ] AI disclosure note updated or deleted at the top of the notebook
- [ ] File renamed: `[LastName]_week11.ipynb`
**Final check:** Run `Kernel → Restart & Run All`. All cells must execute without errors before submitting.
**Submit via Canvas by Sunday 11:59 PM.**